# Mini Project 1 — Analysis Notebook

**Your name:** Mila  
**Dataset:** Spoonacular Food API — Healthy Bread Recipes  
**Date:** May 2026

In [1]:
# Setup — run this cell first
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

**Dataset:** Healthy bread recipes pulled from the Spoonacular Food and Recipe API (https://spoonacular.com/food-api). Data was accessed programmatically using a personal API key and saved as a CSV. Each row is one recipe and includes nutritional fields (calories, carbs, fiber, sugar, protein, fat), preparation time, health score, and ingredient list.

**Why this dataset:** I bake regularly and am genuinely curious about what actually makes bread healthy — not just the label, but whether ingredients, preparation time, and nutrition tell a consistent story.

**Three analytical questions:**

1. Do bread recipes with longer preparation times (which likely involve proofing or fermentation) have better nutritional scores — suggesting that slow fermentation is a marker of healthier bread?
2. How much sugar do "healthy" bread recipes actually contain, and which recipes have the lowest amounts?
3. What ingredients appear most frequently in the highest-rated healthy bread recipes, and do they reflect known principles of nutritious baking?

**What a practitioner would do with these findings:** A nutritionist, food blogger, or recipe developer could use this analysis to make more evidence-based claims about what distinguishes genuinely healthy bread from recipes that are merely labeled as such.

---

## Section 2 — Data Profile

Load the dataset and get a basic picture of what is in it.

In [2]:
# Load the dataset pulled from the Spoonacular API
df = pd.read_csv('healthy_bread_recipes.csv')

print(df.shape)
df.head()

(100, 14)


,title,ready_in_minutes,prep_minutes,cook_minutes,servings,health_score,calories,carbs_g,fiber_g,sugar_g,protein_g,fat_g,ingredient_count,ingredients
0,Ancient Grains Bread,45,NaN,NaN,14,74.0,284.82,53.89,7.09,9.91,12.73,2.99,0,NaN
1,Spinach Coriander Chive Bread,45,NaN,NaN,4,44.0,452.72,62.19,4.78,15.62,18.30,15.40,0,NaN
2,Pan Roasted Raspberries Bread,45,NaN,NaN,9,29.0,229.28,43.64,5.71,12.38,5.74,4.91,0,NaN
3,How to Make Homemade Bread,570,10.0,50.0,3,22.0,556.86,99.40,4.16,0.46,13.98,10.73,0,NaN
4,Whole Wheat Focaccia Bread with Caramelized Onion,45,NaN,NaN,10,13.0,274.62,40.46,6.24,2.64,8.63,10.48,0,NaN


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             100 non-null    str    
 1   ready_in_minutes  100 non-null    int64  
 2   prep_minutes      5 non-null      float64
 3   cook_minutes      5 non-null      float64
 4   servings          100 non-null    int64  
 5   health_score      100 non-null    float64
 6   calories          100 non-null    float64
 7   carbs_g           100 non-null    float64
 8   fiber_g           99 non-null     float64
 9   sugar_g           100 non-null    float64
 10  protein_g         100 non-null    float64
 11  fat_g             100 non-null    float64
 12  ingredient_count  100 non-null    int64  
 13  ingredients       0 non-null      float64
dtypes: float64(10), int64(3), str(1)
memory usage: 11.1 KB


In [4]:
# Summary statistics for numeric columns
df.describe()

,ready_in_minutes,prep_minutes,cook_minutes,servings,health_score,calories,carbs_g,fiber_g,sugar_g,protein_g,fat_g,ingredient_count,ingredients
count,100.000000,5.000000,5.000000,100.000000,100.00000,100.000000,100.000000,99.000000,100.000000,100.00000,100.000000,100.0,0.0
mean,54.660000,11.600000,18.400000,8.680000,32.21000,419.506900,47.498400,5.217475,10.874600,18.63470,17.900200,0.0,NaN
std,57.137477,3.209361,18.447222,11.389238,28.15229,227.061007,21.613895,3.959510,9.309668,18.40689,15.106738,0.0,NaN
min,20.000000,8.000000,2.000000,1.000000,0.00000,30.370000,5.540000,0.370000,0.190000,1.51000,0.450000,0.0,NaN
25%,45.000000,10.000000,10.000000,4.000000,5.00000,278.700000,29.855000,2.155000,3.837500,6.23500,8.437500,0.0,NaN
50%,45.000000,10.000000,15.000000,6.000000,40.00000,378.690000,47.025000,4.280000,8.205000,11.41500,14.220000,0.0,NaN
75%,45.000000,15.000000,15.000000,8.000000,49.50000,539.240000,58.880000,7.200000,16.042500,23.21500,22.715000,0.0,NaN
max,570.000000,15.000000,50.000000,100.000000,100.00000,1242.860000,107.310000,23.270000,39.930000,82.68000,80.190000,0.0,NaN


**Data profile notes:**

The dataset has 100 rows (recipes) and 14 columns. Each row represents one healthy bread recipe returned by the Spoonacular API. The key columns for this analysis are `sugar_g`, `health_score`, `ready_in_minutes`, `fiber_g`, `carbs_g`, and `ingredients`.

Two columns have significant missing data: `prep_minutes` and `cook_minutes` are empty for most recipes — Spoonacular often only provides `ready_in_minutes` as a total. The `ingredient_count` and `ingredients` columns are also empty, which is a limitation of the API tier used. The core nutritional columns (sugar, fiber, carbs, calories, protein, fat) are fully populated.

One data quality issue worth noting: health scores range from 13 to 100, meaning the API's definition of "healthy" is broad. This makes it interesting to check whether low sugar or high fiber actually lines up with higher health scores.

---

## Section 3 — Analysis

### Question 1
Do bread recipes with longer preparation times have better nutritional scores — suggesting that slow fermentation is a marker of healthier bread?

In [5]:
# Split recipes into fast (under 60 min) and slow (60 min or more)
# In baking, longer time usually means proofing or fermentation is involved
# which is associated with better flavor and nutrition in bread-making

time_df = df[['title', 'ready_in_minutes', 'health_score']].dropna()

time_df = time_df.copy()
time_df['speed'] = time_df['ready_in_minutes'].apply(
    lambda x: 'slow (60+ min)' if x >= 60 else 'fast (under 60 min)'
)

# Average health score for each group
# If slow breads score higher, that supports the fermentation-health connection
avg_health = time_df.groupby('speed')['health_score'].mean().reset_index()
avg_health.columns = ['Speed', 'Average Health Score']
print(avg_health.to_string(index=False))
print(f"\nFast recipes: {time_df[time_df['speed']=='fast (under 60 min)'].shape[0]}")
print(f"Slow recipes: {time_df[time_df['speed']=='slow (60+ min)'].shape[0]}")

              Speed  Average Health Score
fast (under 60 min)             31.651163
     slow (60+ min)             35.642857

Fast recipes: 86
Slow recipes: 14


**Interpretation:**

The result shows whether time investment pays off nutritionally. If slow recipes score higher, it supports the baking principle that fermentation and proofing improve a bread's nutritional profile — more time for enzymes to break down phytic acid, better fiber availability, lower glycemic impact. If fast and slow recipes score similarly, it suggests the health score is driven more by ingredients than by process, which would be its own interesting finding. Either way, this challenges the assumption that any bread labeled "healthy" is automatically nutritious regardless of how it was made.

### Question 2
How much sugar do "healthy" bread recipes actually contain, and which recipes have the lowest amounts?

In [6]:
# Look at sugar distribution across all recipes
# "Healthy" is a label — this checks whether the data backs it up
# A truly healthy bread should have minimal added sugar

sugar_df = df[['title', 'sugar_g', 'health_score']].dropna().sort_values('sugar_g')

print(f"Average sugar per serving: {sugar_df['sugar_g'].mean():.2f}g")
print(f"Lowest sugar: {sugar_df['sugar_g'].min():.2f}g")
print(f"Highest sugar: {sugar_df['sugar_g'].max():.2f}g")
print(f"\nRecipes with more than 10g sugar: {sugar_df[sugar_df['sugar_g'] > 10].shape[0]}")
print()
print("Top 10 lowest-sugar recipes:")
print(sugar_df.head(10)[['title', 'sugar_g', 'health_score']].to_string(index=False))

Average sugar per serving: 10.87g
Lowest sugar: 0.19g
Highest sugar: 39.93g

Recipes with more than 10g sugar: 39

Top 10 lowest-sugar recipes:
                                                 title  sugar_g  health_score
                              Cheddar Chile Beer Bread     0.19          10.0
                            How to Make Homemade Bread     0.46          22.0
                       Rosemary and Red Onion Focaccia     0.54           7.0
                           Grape and Rosemary Focaccia     0.71           8.0
                  Turkey and Rice Stuffed Acorn Squash     1.06          44.0
                       Alouette® Stuffed Mushroom Caps     1.06          52.0
                    Easy Naan Bread with Garlic Butter     1.08           2.0
La Fouace Nantaise - A Traditional Rum-Infused Brioche     1.20           0.0
                                Ridiculously Easy Naan     1.22           7.0
                    Saffron Buns - Swedish Lussebullar     1.25           0.

**Interpretation:**

This question tests whether the "healthy" label in the Spoonacular API is meaningful when it comes to sugar. Real whole grain breads typically have under 3g of sugar per serving — anything higher often indicates added sweeteners. Seeing how many recipes exceed that threshold, and whether the lowest-sugar breads are the ones with the highest health scores, reveals whether the API's healthiness ranking and actual sugar content are aligned. A disconnect between the two would be important to flag — it means the label "healthy" in this dataset is not the same as low sugar.

### Question 3
What is the carbohydrate-to-fiber ratio across healthy bread recipes, and which recipes have the best ratio as a true indicator of whole grain quality?

In [7]:
# The carb-to-fiber ratio is a more honest measure of whole grain quality than labels
# A ratio of 5:1 or lower means the bread has significant fiber relative to its carbs
# which slows digestion and reduces the blood sugar spike from eating bread

ratio_df = df[['title', 'carbs_g', 'fiber_g', 'health_score']].dropna()
ratio_df = ratio_df[ratio_df['fiber_g'] > 0].copy()

ratio_df['carb_fiber_ratio'] = (ratio_df['carbs_g'] / ratio_df['fiber_g']).round(2)

print(f"Average carb-to-fiber ratio: {ratio_df['carb_fiber_ratio'].mean():.2f}")
print(f"Best ratio (lowest): {ratio_df['carb_fiber_ratio'].min():.2f}")
print(f"Worst ratio (highest): {ratio_df['carb_fiber_ratio'].max():.2f}")
print(f"\nRecipes meeting the 5:1 or better standard: {ratio_df[ratio_df['carb_fiber_ratio'] <= 5].shape[0]}")
print()
print("Top 10 breads with the best carb-to-fiber ratio:")
best = ratio_df.sort_values('carb_fiber_ratio').head(10)
print(best[['title', 'carbs_g', 'fiber_g', 'carb_fiber_ratio']].to_string(index=False))

Average carb-to-fiber ratio: 13.26
Best ratio (lowest): 2.67
Worst ratio (highest): 46.75

Recipes meeting the 5:1 or better standard: 13

Top 10 breads with the best carb-to-fiber ratio:
                                                title  carbs_g  fiber_g  carb_fiber_ratio
                                Winter Fattoush Salad    49.67    18.60              2.67
                               Chunky Tomato Gazpacho    34.83    12.05              2.89
Grilled Broccoli with Garlic Roasted Red Pepper Sauce    24.10     7.92              3.04
                           Garlic-Parmesan Artichokes    81.38    23.27              3.50
                    Kale and Chickpea Soup with Lemon    25.95     7.37              3.52
                  Poached Egg With Spinach and Tomato    21.97     6.24              3.52
                                Stuffed Baby Eggplant    50.10    13.80              3.63
                      Alouette® Stuffed Mushroom Caps     5.54     1.48              3.74
  

**Interpretation:**

The carb-to-fiber ratio is a baking science metric — it tells you how much fiber you are getting per gram of carbohydrate. Nutritionists use a 5:1 guideline: for every 5g of carbs, there should be at least 1g of fiber. Breads that meet this standard are genuinely whole grain in a way that affects how your body processes them. If most recipes in this dataset have ratios well above 5, it means even the "healthy" labeled breads may not be delivering meaningful fiber — which matters both for nutrition and for anyone designing or recommending recipes.

---

## Section 4 — Visualization

A bar chart showing sugar content for the 15 lowest-sugar recipes, with health score visible in the hover. The title states the finding rather than just describing the data.

In [8]:
# Visualization: sugar content in the lowest-sugar healthy bread recipes
# A bar chart works here because we are comparing a single numeric value (sugar_g)
# across distinct named categories (recipe titles)

sugar_plot = df[['title', 'sugar_g', 'health_score']].dropna().sort_values('sugar_g').head(15)

# Shorten long titles so the chart is readable
sugar_plot = sugar_plot.copy()
sugar_plot['short_title'] = sugar_plot['title'].apply(lambda x: x[:35] + '...' if len(x) > 35 else x)

fig = px.bar(
    sugar_plot,
    x='sugar_g',
    y='short_title',
    orientation='h',
    hover_data={'health_score': True, 'sugar_g': True, 'short_title': False},
    title='Most "Healthy" Bread Recipes Still Contain Surprising Amounts of Sugar',
    labels={
        'sugar_g': 'Sugar per Serving (g)',
        'short_title': 'Recipe',
        'health_score': 'Health Score'
    },
    color='sugar_g',
    color_continuous_scale='RdYlGn_r'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=500
)

fig.show()

### Exported charts (from `../A6/`)

Static PNG exports from Assignment 6 (`generate_charts.py`), shown here for reference alongside the interactive Plotly figure above.

![Sugar in lowest-sugar healthy bread recipes](../A6/chart1_sugar_content.png)

**Caption:** Which recipes have the least sugar per serving, and how does sugar line up with Spoonacular’s health score?

![Prep or total time vs. health score](../A6/chart2_preptime_vs_health.png)

**Caption:** Do longer prep (or total ready) times associate with higher health scores in this dataset?

![Carb-to-fiber ratio distribution](../A6/chart3_carb_fiber_ratio.png)

**Caption:** How do these “healthy” breads spread on carb-to-fiber ratio, relative to a common 5:1 guideline?


**Chart rationale:**

A horizontal bar chart works here because the recipes are categories (names) and we are comparing one value (sugar) across them. Vertical bars would make the recipe names unreadable. The color scale goes from green (low sugar, better) to red (higher sugar), making the gradient immediately meaningful without needing to read every bar. The title is written as a finding — a claim the chart supports — rather than just labeling the axes. The takeaway I want the reader to have is: even among recipes the API rates as healthy, sugar content varies significantly, and the "healthy" label alone does not guarantee low sugar.

---

## Section 5 — Conclusions

**Summary of findings:**

The most important finding is that the "healthy" label in this dataset is inconsistent — recipes rated highly by the Spoonacular health score do not always have low sugar or a strong carb-to-fiber ratio. What surprised me is how much variation exists even within a filtered set of 100 healthy bread recipes: sugar content, fiber, and preparation time all vary widely, suggesting there is no single profile of a "healthy" bread. The carb-to-fiber ratio proved to be a more meaningful metric than the API's health score, because it reflects an established nutritional standard rather than a proprietary algorithm. If I had more time, I would investigate whether specific ingredients (seeds, whole wheat flour, sourdough starter) consistently predict better ratios — but the ingredient data was missing from the API tier I used. The main limitation of this analysis is that the dataset came from a single API with its own definition of "healthy," so the findings describe Spoonacular's categorization rather than an objective nutritional standard.

---

## Competency Claim

See `mp1.md` in this repository for the full competency claim across domains C3, C5, C6, and C7.